# AIMO E2.6 — Early Dynamics Validation

Run on Kaggle GPU. Compares **E1 14D / E2A-Full 26D / E2A-Slope 18D** at **128/256/512/1024** generated-token budgets. One deterministic generation to 1024 tokens is reused for every prefix. Validation: 10-seed grouped CV + 5-seed nested grouped CV. The final cells evaluate Gate A (early signal), Gate B (compact slope), and Gate C (early plateau), then package outputs.


In [ ]:
from pathlib import Path
import os, sys, json, shutil, zipfile, subprocess, platform
import numpy as np
import pandas as pd

REPO_SOURCE="github"  # or "dataset_snapshot"
REPO_URL="https://github.com/luxury221/getting-started.git"
BRANCH="research/temporal-uncertainty-v1"
REPO_ZIP_OVERRIDE=None
MODEL_ID="deepseek-ai/DeepSeek-R1-0528-Qwen3-8B"
PREFIX_BUDGETS=[128,256,512,1024]
MAX_SAMPLES=64
BATCH_SIZE=1
OFFICIAL_SEEDS=list(range(10))
NESTED_SEEDS=list(range(5))
MAX_OUTER_SPLITS=5
INNER_SPLITS=4
EARLY_MAX_BUDGET=512
GATE_A_MIN_DELTA=0.03
GATE_A_MIN_POSITIVE=4
GATE_B_MAX_GAP=0.01
GATE_B_MIN_RECOVERY=0.80
GATE_C_MAX_GAP=0.02

WORK=Path("/kaggle/working"); REPO=WORK/"getting-started"
OUT=WORK/"aimo_e26_outputs"; OUT.mkdir(parents=True,exist_ok=True)
HF=WORK/"hf-cache"; HF.mkdir(parents=True,exist_ok=True)
FEATURE=OUT/f"early_dynamics_{MAX_SAMPLES}.parquet"
OFFICIAL_JSON=OUT/"early_dynamics_official.json"; OFFICIAL_CSV=OUT/"early_dynamics_official.csv"
NESTED_JSON=OUT/"early_dynamics_nested.json"; NESTED_CSV=OUT/"early_dynamics_nested.csv"
DECISION=OUT/"e2_6_decision.json"
print("Python",sys.version); subprocess.run(["nvidia-smi"],check=False)


In [ ]:
pkgs=["accelerate==1.13.0","huggingface-hub==1.22.0","joblib==1.5.3","pandas==3.0.3","safetensors==0.8.0","scikit-learn==1.8.0","tokenizers==0.22.2","transformers==5.13.0","pyarrow>=17"]
subprocess.run([sys.executable,"-m","pip","install","-q",*pkgs],check=True)

if REPO.exists(): shutil.rmtree(REPO)
if REPO_SOURCE=="github":
    subprocess.run(["git","clone","--depth","1","--branch",BRANCH,REPO_URL,str(REPO)],check=True)
    repo_version=subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip()
else:
    if REPO_ZIP_OVERRIDE: z=Path(REPO_ZIP_OVERRIDE)
    else:
        hits=sorted(Path("/kaggle/input").rglob("getting-started*.zip"))
        if not hits: raise FileNotFoundError("Upload getting-started*.zip or use REPO_SOURCE='github'")
        z=hits[0]
    tmp=WORK/"_e26_repo"; shutil.rmtree(tmp,ignore_errors=True); tmp.mkdir()
    with zipfile.ZipFile(z) as f: f.extractall(tmp)
    roots=[p for p in [tmp,*tmp.rglob("*")] if p.is_dir() and (p/"solutions/uncertainty-profiling/scripts").exists()]
    if not roots: raise RuntimeError("Repository root not found inside ZIP")
    shutil.copytree(sorted(roots,key=lambda p:len(p.parts))[0],REPO)
    shutil.rmtree(tmp,ignore_errors=True); repo_version=f"dataset-snapshot:{z.name}"

SOL=REPO/"solutions/uncertainty-profiling"; SCRIPTS=SOL/"scripts"; TESTS=SOL/"tests"
for p in [SCRIPTS/"compute_temporal_features.py",SCRIPTS/"run_early_exit_validation.py",TESTS/"test_early_exit_schema.py"]:
    assert p.exists(),p
print("repo_version",repo_version)
subprocess.run([sys.executable,"-m","unittest","discover",str(TESTS),"-p","test_early_exit_schema.py"],cwd=REPO,check=True)
print("schema tests PASS")


In [ ]:
cmd=[sys.executable,str(SCRIPTS/"compute_temporal_features.py"),
     "--feature-model-id",MODEL_ID,"--cache-dir",str(HF),
     "--prefix-budgets",*[str(x) for x in PREFIX_BUDGETS],
     "--max-new-tokens",str(max(PREFIX_BUDGETS)),
     "--max-samples",str(MAX_SAMPLES),"--batch-size",str(BATCH_SIZE),
     "--output",str(FEATURE),"--overwrite"]
print(" ".join(cmd)); subprocess.run(cmd,cwd=REPO,check=True)

df=pd.read_parquet(FEATURE)
groups=df[["original_problem","model_is_robust"]].drop_duplicates("original_problem")
counts=groups["model_is_robust"].value_counts()
if len(counts)!=2: raise RuntimeError("Need both robustness classes")
SAFE_SPLITS=min(MAX_OUTER_SPLITS,int(counts.min()))
if SAFE_SPLITS<2: raise RuntimeError("Too few groups per class")
print("rows",len(df),"groups",df["original_problem"].nunique(),"SAFE_SPLITS",SAFE_SPLITS)
for b in PREFIX_BUDGETS:
    c=pd.to_numeric(df[f"prefix_{b}__num_tokens"],errors="coerce").to_numpy(float)
    print(b,"mean_tokens",c.mean(),"reached",np.mean(c>=b))


In [ ]:
def run(protocol,seeds,out_json,out_csv):
    cmd=[sys.executable,str(SCRIPTS/"run_early_exit_validation.py"),
         "--feature-data-path",str(FEATURE),
         "--budgets",*[str(x) for x in PREFIX_BUDGETS],
         "--protocol",protocol,"--n-splits",str(SAFE_SPLITS)]
    if protocol=="nested": cmd += ["--inner-splits",str(INNER_SPLITS)]
    cmd += ["--seeds",*[str(s) for s in seeds],"--results-path",str(out_json),"--summary-csv",str(out_csv)]
    print(" ".join(cmd)); subprocess.run(cmd,cwd=REPO,check=True)

run("official",OFFICIAL_SEEDS,OFFICIAL_JSON,OFFICIAL_CSV)
run("nested",NESTED_SEEDS,NESTED_JSON,NESTED_CSV)
official=pd.read_csv(OFFICIAL_CSV); nested=pd.read_csv(NESTED_CSV)
display(official); display(nested)


In [ ]:
import matplotlib.pyplot as plt
for name,t in [("Official",official),("Nested",nested)]:
    fig,ax=plt.subplots(figsize=(8,5))
    ax.plot(t["budget"],t["e1_ba_mean"],marker="o",label="E1 14D")
    ax.plot(t["budget"],t["e2a_full_ba_mean"],marker="o",label="E2A Full 26D")
    ax.plot(t["budget"],t["e2a_slope_ba_mean"],marker="o",label="E2A Slope 18D")
    ax.set(xlabel="Generated-token budget",ylabel="Balanced Accuracy",title=f"E2.6 Early Dynamics — {name}")
    ax.legend(); plt.tight_layout(); plt.show()


In [ ]:
with open(NESTED_JSON,encoding="utf-8") as f: payload=json.load(f)
full1024=float(nested.loc[nested.budget==max(PREFIX_BUDGETS),"e2a_full_ba_mean"].iloc[0])

A=[]; B=[]; C=[]
for _,r in nested.iterrows():
    b=int(r["budget"])
    if b>EARLY_MAX_BUDGET: continue
    one=payload["results"][str(b)]
    e1={int(x["seed"]):float(x["balanced_accuracy"]) for x in one["E1_official_14d"]["per_seed"]}
    fu={int(x["seed"]):float(x["balanced_accuracy"]) for x in one["E2A_full_26d"]["per_seed"]}
    common=sorted(set(e1)&set(fu)); pos=sum(fu[s]-e1[s]>0 for s in common)
    delta=float(r["full_delta_ba"])
    A.append({"budget":b,"delta_ba":delta,"positive_seeds":pos,"passes":delta>=GATE_A_MIN_DELTA and pos>=GATE_A_MIN_POSITIVE})
    full=float(r["e2a_full_ba_mean"]); slope=float(r["e2a_slope_ba_mean"])
    rec=float(r["slope_gain_recovery_ratio"]) if np.isfinite(r["slope_gain_recovery_ratio"]) else np.nan
    gap=full-slope
    B.append({"budget":b,"full_ba":full,"slope_ba":slope,"gap":gap,"recovery":rec,"passes":gap<=GATE_B_MAX_GAP or (np.isfinite(rec) and rec>=GATE_B_MIN_RECOVERY)})
    pgap=full1024-full
    C.append({"budget":b,"full_ba":full,"full1024_ba":full1024,"gap_to_1024":pgap,"passes":pgap<=GATE_C_MAX_GAP})

ga=any(x["passes"] for x in A); gb=any(x["passes"] for x in B); gc=any(x["passes"] for x in C)
decision="STRONG_GO_REPLICATE_128" if ga and gb and gc else ("GO_REPLICATE_128" if ga else "MODIFY_BEFORE_PHASE_II")
print("DECISION:",decision)
print("Gate A"); display(pd.DataFrame(A))
print("Gate B"); display(pd.DataFrame(B))
print("Gate C"); display(pd.DataFrame(C))


In [ ]:
def earliest(xs):
    ys=[x["budget"] for x in xs if x["passes"]]
    return min(ys) if ys else None
result={
 "experiment":"E2.6 Early Dynamics Validation","decision":decision,
 "repo_version":repo_version,"model_id":MODEL_ID,"max_samples":MAX_SAMPLES,
 "prefix_budgets":PREFIX_BUDGETS,"safe_outer_splits":SAFE_SPLITS,
 "official_seeds":OFFICIAL_SEEDS,"nested_seeds":NESTED_SEEDS,
 "gates":{
   "A":{"pass":ga,"earliest_budget":earliest(A),"candidates":A},
   "B":{"pass":gb,"earliest_budget":earliest(B),"candidates":B},
   "C":{"pass":gc,"earliest_budget":earliest(C),"candidates":C}
 }}
with open(DECISION,"w",encoding="utf-8") as f: json.dump(result,f,indent=2,ensure_ascii=False)
print(json.dumps(result,indent=2,ensure_ascii=False))
archive=shutil.make_archive("/kaggle/working/aimo_e2_6_results","zip",OUT)
print("ZIP:",archive)


## Output

Send `/kaggle/working/aimo_e2_6_results.zip` back for analysis. Keep `early_dynamics_64.parquet`: all CV/gating analyses can be rerun without repeating GPU generation. If the result is `STRONG_GO_REPLICATE_128` or `GO_REPLICATE_128`, rerun with `MAX_SAMPLES=128` before moving to Representation Dynamics.
